<a href="https://colab.research.google.com/github/MightyCrimsonX/Crimson-Notebooks/blob/main/Notebooks/Dataset_Maker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Dataset Maker
## Whitez Fork / Modified by Mighty Crimson
This is based on the work of [Kohya-ss](https://github.com/kohya-ss/sd-scripts) and [Linaqruf](https://colab.research.google.com/github/Linaqruf/kohya-trainer/blob/main/kohya-LoRA-dreambooth.ipynb). Thank you!

### ⭕ Disclaimer
The purpose of this document is to research bleeding-edge technologies in the field of machine learning inference.  
Please read and follow the [Google Colab guidelines](https://research.google.com/colaboratory/faq.html) and its [Terms of Service](https://research.google.com/colaboratory/tos_v3.html).

| |GitHub|MightyCrimson Lora Trainer SDXL|MightyCrimson EasyTrainerScript|Anima Kohya Trainer|Dataset Maker|
|:--|:-:|:-:|:-:|:-:|:-:|
| 🏠 **Original Project** | [![GitHub](https://raw.githubusercontent.com/hollowstrawberry/kohya-colab/main/assets/github.svg )](https://github.com/hollowstrawberry/kohya-colab ) | | | | |
| **Modified By mightyCrimson** | [![GitHub](https://raw.githubusercontent.com/hollowstrawberry/kohya-colab/main/assets/github.svg )](https://github.com/gwhitez/Lora-Trainer-XL ) | [![Open in Colab](https://raw.githubusercontent.com/hollowstrawberry/kohya-colab/main/assets/colab-badge.svg )](https://colab.research.google.com/drive/1F2DtUZSc8NPetD4558bf4VdLQ7uyTTvM ) | [![Open in Colab](https://raw.githubusercontent.com/hollowstrawberry/kohya-colab/main/assets/colab-badge.svg )](https://colab.research.google.com/github/MightyCrimsonX/Kohya-Anima-Trainer-Mighty/blob/main/MightyCrimson_EasyTrainerScript.ipynb ) | [![Open in Colab](https://raw.githubusercontent.com/hollowstrawberry/kohya-colab/main/assets/colab-badge.svg )](https://colab.research.google.com/github/MightyCrimsonX/Kohya-Anima-Trainer-Mighty/blob/main/MightyCrimson_LoraAnimaTrainer.ipynb) | [![Open in Colab](https://raw.githubusercontent.com/hollowstrawberry/kohya-colab/main/assets/colab-badge.svg )](https://colab.research.google.com/drive/1F2DtUZSc8NPetD4558bf4VdLQ7uyTTvM ) |

In [ ]:
import os
from IPython import get_ipython
from IPython.display import display, Markdown

COLAB = True

if COLAB:
  from google.colab.output import clear as clear_output
else:
  from IPython.display import clear_output

#@title ## 🚩 Start Here

#@markdown ### 1️⃣ Setup
#@markdown This cell will load some requirements and create the necessary folders in your Google Drive. <p>
#@markdown Your project name can't contain spaces but it can contain a single / to make a subfolder in your dataset.
project_name = "" #@param {type:"string"}
project_name = project_name.strip()
#@markdown The folder structure doesn't matter and is purely for comfort. Make sure to always pick the same one. I like organizing by project.
folder_structure = "Organize by project (MyDrive/Loras/project_name/dataset)" #@param ["Organize by category (MyDrive/lora_training/datasets/project_name)", "Organize by project (MyDrive/Loras/project_name/dataset)"]

if not project_name or any(c in project_name for c in " .()\"'\\") or project_name.count("/") > 1:
  print("Please write a valid project_name.")
else:
  if COLAB and not os.path.exists('/content/drive'):
    from google.colab import drive
    print("📂 Connecting to Google Drive...")
    drive.mount('/content/drive')

  project_base = project_name if "/" not in project_name else project_name[:project_name.rfind("/")]
  project_subfolder = project_name if "/" not in project_name else project_name[project_name.rfind("/")+1:]

  root_dir = "/content" if COLAB else "~/Loras"
  deps_dir = os.path.join(root_dir, "deps")

  if "/Loras" in folder_structure:
    main_dir      = os.path.join(root_dir, "drive/MyDrive/Loras") if COLAB else root_dir
    config_folder = os.path.join(main_dir, project_base)
    images_folder = os.path.join(main_dir, project_base, "dataset")
    if "/" in project_name:
      images_folder = os.path.join(images_folder, project_subfolder)
  else:
    main_dir      = os.path.join(root_dir, "drive/MyDrive/lora_training") if COLAB else root_dir
    config_folder = os.path.join(main_dir, "config", project_name)
    images_folder = os.path.join(main_dir, "datasets", project_name)

  for dir in [main_dir, deps_dir, images_folder, config_folder]:
    os.makedirs(dir, exist_ok=True)

  print(f"✅ Project {project_name} is ready!")
  step1_installed_flag = True


In [ ]:
if "step1_installed_flag" not in globals():
  raise Exception("Please run step 1 first!")

#@markdown ### 4️⃣ Tag your images
#@markdown We will be using AI to automatically tag your images, specifically [Waifu Diffusion](https://huggingface.co/SmilingWolf/wd-eva02-large-tagger-v3) in the case of anime and [BLIP](https://huggingface.co/spaces/Salesforce/BLIP) in the case of photos. <p>
#@markdown Giving tags/captions to your images allows for much better training. This process takes 5 minutes to install and 5 more minutes to tag a thousand images. It goes through all subfolders if you have any. <p>
method = "Anime tags" #@param ["Anime tags", "Photo captions"]
#@markdown **Anime** New Tagger Models: `pixai-labs/pixai-tagger-v1.0`, `ashen-sensored/wd-eva02-tagger-2026-canary`.
tagger = "ashen-sensored/wd-eva02-tagger-2026-canary" # @param ["pixai-labs/pixai-tagger-v1.0", "ashen-sensored/wd-eva02-tagger-2026-canary", "SmilingWolf/wd-eva02-large-tagger-v3", "SmilingWolf/wd-vit-large-tagger-v3", "Both"]
tag_threshold = 0.25 #@param {type:"slider", min:0.0, max:1.0, step:0.01}
blacklist_tags = "virtual youtuber, parody, style parody, official alternate costume, official alternate hairstyle, official alternate hair length, alternate costume, alternate hairstyle, alternate hair length, alternate hair color" #@param {type:"string"}
include_character_names = False #@param {type:"boolean"}
#@markdown **Photos:** The minimum and maximum length of tokens/words in each caption.
caption_min = 10 #@param {type:"number"}
caption_max = 75 #@param {type:"number"}

character_threshold = tag_threshold if include_character_names else 1.1
undesired_tags = '"' + ','.join([t.strip() for t in blacklist_tags.split(",") if t.strip()]) + '"'
undesired_tags_list = [t.strip() for t in blacklist_tags.split(",") if t.strip()]

import os

kohya_dir = "/content/kohya"
venv_python = os.path.join(kohya_dir, ".venv/bin/python")
# venv_pip = os.path.join(kohya_dir, "venv/bin/pip")

# Models that use timm + safetensors (not ONNX/TF)
TIMM_TAGGER_MODELS = {
  "ashen-sensored/wd-eva02-tagger-2026-canary",
}

# PixAI Tagger models (Hugging Face Transformers pipeline with SAM3 backbone)
PIXAI_TAGGER_MODELS = {
  "pixai-labs/pixai-tagger-v1.0",
}

if "step4_installed_flag" not in globals():
  print("starting ⭐")
#install uv
  !curl -LsSf https://astral.sh/uv/install.sh | sh

  !git clone https://github.com/kohya-ss/sd-scripts {kohya_dir}
  os.chdir(kohya_dir)

  !uv venv --python 3.10 --clear

  !uv pip install --python {venv_python} \
    -r requirements.txt --no-progress \
    fairscale==0.4.13 \
    "timm>=0.9.16" \
    safetensors \
    onnx \
    onnxruntime-gpu

  # installing rich
  !uv pip uninstall --python {venv_python} rich

  # Install dependencies in notebook kernel for PixAI / timm taggers
  !pip -q install "timm>=0.9.16" "transformers>=4.40.0" accelerate

  step4_installed_flag = True
  print("\n✅ Dependencies installed successfully.\n")

print("\n🚶‍♂️ Launching program...\n")
os.chdir(kohya_dir)



def run_timm_tagger(repo_id, images_folder, tag_threshold, character_threshold,
                    undesired_tags_list, append_tags=False, remove_underscore=True, recursive=True):
  """
  Run a timm/safetensors WD-style tagger (e.g. ashen-sensored/wd-eva02-tagger-2026-canary).
  Uses the EXACT preprocessing of the official HuggingFace Space:
    - White background compositing (RGBA -> RGB)
    - Pad to square
    - Resize to model input size with BICUBIC
    - BGR channel order
    - Normalize to [-1, 1] (pixel / 127.5 - 1.0)
  """
  import csv
  import timm
  import torch
  import numpy as np
  from PIL import Image
  from pathlib import Path
  from huggingface_hub import hf_hub_download
  from safetensors.torch import load_file

  # Per-model configuration
  TIMM_MODEL_CONFIG = {
    "ashen-sensored/wd-eva02-tagger-2026-canary": {
      "timm_arch": "eva02_large_patch14_448",
      "input_size": 448,
      "model_file": "model.safetensors",
    },
  }
  cfg = TIMM_MODEL_CONFIG.get(repo_id, {
    "timm_arch": "eva02_large_patch14_448",
    "input_size": 448,
    "model_file": "model.safetensors",
  })
  input_size = cfg["input_size"]

  # Download model weights and tags CSV
  print(f"  🔽 Downloading model files from: {repo_id}")
  csv_path = hf_hub_download(repo_id=repo_id, filename="selected_tags.csv")
  model_path = hf_hub_download(repo_id=repo_id, filename=cfg["model_file"])

  # Load tags from CSV (preserving row order = model output order)
  import pandas as pd
  tags_df = pd.read_csv(csv_path)
  kaomojis = {"0_0","(o)_(o)","+_+","+_-","._.",">o>_>o>","<|>_<|>","=_=",">_<","3_3","6_9",">_o","@_@","^_^","o_o","u_u","x_x","|_|","||_||"}
  tag_names = tags_df["name"].map(
    lambda x: x.replace("_", " ") if x not in kaomojis else x
  ).tolist()
  general_indexes  = list(np.where(tags_df["category"] == 0)[0])
  character_indexes = list(np.where(tags_df["category"] == 4)[0])

  # Build the undesired set (normalised: underscores already replaced above)
  undesired_set = set(undesired_tags_list)
  # Also add underscore-form in case tags_list was built before replacement
  undesired_set.update(t.replace(" ", "_") for t in undesired_tags_list)

  # Load model architecture + weights from safetensors
  print(f"  🏗️  Loading model architecture: {cfg['timm_arch']}")
  model = timm.create_model(
    cfg["timm_arch"],
    pretrained=False,
    num_classes=len(tag_names),
  )
  model.load_state_dict(load_file(model_path), strict=True)
  model = model.eval()
  if torch.cuda.is_available():
    model = model.cuda()

  def prepare_image(image):
    """Exact preprocessing from the official Space app.py."""
    # White background compositing (handles RGBA/transparency)
    canvas = Image.new("RGBA", image.size, (255, 255, 255))
    canvas.alpha_composite(image.convert("RGBA"))
    image = canvas.convert("RGB")

    # Pad to square
    w, h = image.size
    max_dim = max(w, h)
    pad_left = (max_dim - w) // 2
    pad_top  = (max_dim - h) // 2
    padded = Image.new("RGB", (max_dim, max_dim), (255, 255, 255))
    padded.paste(image, (pad_left, pad_top))

    # Resize to model input size with BICUBIC
    if max_dim != input_size:
      padded = padded.resize((input_size, input_size), Image.BICUBIC)

    # Convert to float32 numpy, flip RGB→BGR, normalize to [-1, 1]
    arr = np.asarray(padded, dtype=np.float32)
    arr = arr[:, :, ::-1] / 127.5 - 1.0   # BGR + [-1,1]

    return torch.from_numpy(arr.copy()).permute(2, 0, 1).unsqueeze(0)

  # Collect image paths
  if recursive:
    image_paths = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
      image_paths.extend(Path(images_folder).rglob(ext))
  else:
    image_paths = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
      image_paths.extend(Path(images_folder).glob(ext))

  print(f"  📸 Processing {len(image_paths)} images with {repo_id}...")
  for img_path in image_paths:
    try:
      image = Image.open(img_path)
      tensor = prepare_image(image)
      if torch.cuda.is_available():
        tensor = tensor.cuda()

      with torch.no_grad():
        preds = torch.sigmoid(model(tensor)[0]).cpu().numpy()

      # Map output indices to tag names (same as Space)
      result_tags = []

      # Character tags first
      for idx in character_indexes:
        if preds[idx] >= character_threshold:
          tag = tag_names[idx]
          if tag not in undesired_set and tag.replace(" ", "_") not in undesired_set:
            result_tags.append(tag)

      # General tags
      for idx in general_indexes:
        if preds[idx] >= tag_threshold:
          tag = tag_names[idx]
          if tag not in undesired_set and tag.replace(" ", "_") not in undesired_set:
            result_tags.append(tag)

      txt_path = img_path.with_suffix(".txt")
      if append_tags and txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
          existing = f.read().strip()
        existing_tags = [t.strip() for t in existing.split(",") if t.strip()]
        for t in result_tags:
          if t not in existing_tags:
            existing_tags.append(t)
        result_tags = existing_tags

      with open(txt_path, "w", encoding="utf-8") as f:
        f.write(", ".join(result_tags))
    except Exception as e:
      print(f"  ⚠️ Error processing {img_path.name}: {e}")

  print(f"  ✅ Done tagging with {repo_id}")


def run_pixai_tagger(images_folder, tag_threshold=0.5, include_character_names=True,
                     undesired_tags_list=None, append_tags=False, remove_underscore=True, recursive=True):
  """
  Run PixAI Tagger v1.0 (SAM3 backbone at 1008x1008, 30k vocabulary).
  - Rating, copyright, and meta tags: DEACTIVATED / excluded.
  - Character threshold: 0.27 if include_character_names is True, else omitted.
  - Style threshold: 0.15 by default.
  - General threshold: uses tag_threshold (or recommended 0.17 if tag_threshold is left at default 0.5).
  """
  import torch
  from PIL import Image
  from pathlib import Path
  from transformers import pipeline

  model_id = "pixai-labs/pixai-tagger-v1.0"
  print(f"  🔽 Loading PixAI Tagger model: {model_id}...")

  # General threshold: if user kept default 0.5, use PixAI recommended 0.17; otherwise user value
  gen_threshold = tag_threshold if tag_threshold != 0.5 else 0.17
  char_threshold = 0.27 if include_character_names else 1.1
  style_threshold = 0.15

  print(f"  ⚙️ Active Thresholds -> General: {gen_threshold:.2f} | Character: {char_threshold:.2f} (Included: {include_character_names}) | Style: {style_threshold:.2f}")
  print("  🚫 Rating, Copyright, and Meta tagging are completely deactivated.")

  device = 0 if torch.cuda.is_available() else -1
  tagger_pipe = pipeline(
    model=model_id,
    image_processor=model_id,
    trust_remote_code=True,
    device=device,
  )

  # Build undesired tags set
  undesired_set = set(undesired_tags_list or [])
  undesired_set.update(t.replace(" ", "_") for t in (undesired_tags_list or []))

  # Collect image paths
  if recursive:
    image_paths = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
      image_paths.extend(Path(images_folder).rglob(ext))
  else:
    image_paths = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
      image_paths.extend(Path(images_folder).glob(ext))

  print(f"  📸 Processing {len(image_paths)} images with PixAI Tagger v1.0...")
  for img_path in image_paths:
    try:
      raw_img = Image.open(img_path)
      # Handle transparency / alpha channels with white background
      if raw_img.mode in ("RGBA", "LA") or (raw_img.mode == "P" and "transparency" in raw_img.info):
        canvas = Image.new("RGBA", raw_img.size, (255, 255, 255))
        canvas.alpha_composite(raw_img.convert("RGBA"))
        image = canvas.convert("RGB")
      else:
        image = raw_img.convert("RGB")

      # Run inference with category-specific thresholds
      # Setting copyright, meta, rating to 1.1 completely suppresses them
      pipe_out = tagger_pipe(
        image,
        threshold={
          "general": gen_threshold,
          "character": char_threshold,
          "style": style_threshold,
          "copyright": 1.1,
          "meta": 1.1,
          "rating": 1.1,
        },
      )
      results = pipe_out.get("results", {})

      result_tags = []

      # 1. Character tags first (only if enabled)
      if include_character_names and "character" in results:
        for tag in results["character"].keys():
          tag_clean = tag.replace("_", " ") if remove_underscore else tag
          if tag_clean not in undesired_set and tag_clean.replace(" ", "_") not in undesired_set:
            result_tags.append(tag_clean)

      # 2. General tags
      if "general" in results:
        for tag in results["general"].keys():
          tag_clean = tag.replace("_", " ") if remove_underscore else tag
          if tag_clean not in undesired_set and tag_clean.replace(" ", "_") not in undesired_set:
            result_tags.append(tag_clean)

      # 3. Style tags
      if "style" in results:
        for tag in results["style"].keys():
          tag_clean = tag.replace("_", " ") if remove_underscore else tag
          if tag_clean not in undesired_set and tag_clean.replace(" ", "_") not in undesired_set:
            result_tags.append(tag_clean)

      # Note: Rating, Copyright, and Meta are never appended

      txt_path = img_path.with_suffix(".txt")
      if append_tags and txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
          existing = f.read().strip()
        existing_tags = [t.strip() for t in existing.split(",") if t.strip()]
        for t in result_tags:
          if t not in existing_tags:
            existing_tags.append(t)
        result_tags = existing_tags

      with open(txt_path, "w", encoding="utf-8") as f:
        f.write(", ".join(result_tags))

    except Exception as e:
      print(f"  ⚠️ Error processing {img_path.name}: {e}")

  print("  ✅ Done tagging with PixAI Tagger v1.0")

if "Anime" in method:
  # Determine the list of models to run
  if tagger == "Both":
    tagger_models = [
      "SmilingWolf/wd-eva02-large-tagger-v3",
      "SmilingWolf/wd-vit-large-tagger-v3"
    ]
  else:
    tagger_models = [tagger]

  for i, tagger_model in enumerate(tagger_models):
    is_append = i > 0
    print(f"\n🏷️ Running tagger [{i+1}/{len(tagger_models)}]: {tagger_model}")

    if tagger_model in PIXAI_TAGGER_MODELS:
      # Use the PixAI Tagger v1.0 pipeline (SAM3 backbone, 1008x1008)
      run_pixai_tagger(
        images_folder=images_folder,
        tag_threshold=tag_threshold,
        include_character_names=include_character_names,
        undesired_tags_list=undesired_tags_list,
        append_tags=is_append,
        remove_underscore=True,
        recursive=True,
      )
    elif tagger_model in TIMM_TAGGER_MODELS:
      # Use the timm/safetensors inference path
      run_timm_tagger(
        repo_id=tagger_model,
        images_folder=images_folder,
        tag_threshold=tag_threshold,
        character_threshold=character_threshold,
        undesired_tags_list=undesired_tags_list,
        append_tags=is_append,
        remove_underscore=True,
        recursive=True,
      )
    else:
      # Use the kohya ONNX inference path
      append_flag = "--append_tags" if is_append else ""
      !{venv_python} finetune/tag_images_by_wd14_tagger.py \
        {images_folder} \
        --repo_id={tagger_model} \
        --general_threshold={tag_threshold} \
        --character_threshold={character_threshold} \
        --batch_size=8 \
        --max_data_loader_n_workers=2 \
        --caption_extension=.txt \
        --undesired_tags {undesired_tags} \
        --onnx --recursive --remove_underscore {append_flag}

  # Count and display top tags across all taggers
  from collections import Counter
  text_files = []
  for root, dirs, files in os.walk(images_folder):
    for file in files:
      if file.lower().endswith(".txt"):
        text_files.append(os.path.join(root, file))
  top_tags = Counter()
  for file in text_files:
    with open(file, 'r') as f:
      tags = [t.strip() for t in f.read().split(",")]
    top_tags.update(tags)

  clear_output()
  print(f"📊 Tagging complete. Here are the top 50 tags in your dataset:")
  print("\n".join(f"{k} ({v})" for k, v in top_tags.most_common(50)))

else:
  !{venv_python} finetune/make_captions.py \
    {images_folder} \
    --beam_search \
    --max_data_loader_n_workers=2 \
    --batch_size=8 \
    --min_length={caption_min} \
    --max_length={caption_max} \
    --caption_extension=.txt \
    --recursive

  if not get_ipython().__dict__['user_ns']['_exit_code']:
    import random
    text_files = []
    for root, dirs, files in os.walk(images_folder):
      for file in files:
        if file.lower().endswith(".txt"):
          text_files.append(os.path.join(root, file))
    sample = []
    for file in text_files:
      with open(file, 'r') as f:
        sample.append(f.read())

    clear_output()
    print(f"📊 Captioning complete. Here are {len(sample)} example captions from your dataset:")
    print("".join(sample))

os.chdir(root_dir)


In [ ]:
if "step1_installed_flag" not in globals():
  raise Exception("Please run step 1 first!")

#@markdown ### 5️⃣ Curate your tags
#@markdown Modify your dataset's tags. You can run this cell multiple times with different parameters. <p>

#@markdown Put an activation tag at the start of every text file. This is useful to make learning better and activate your Lora easier. Set `keep_tokens` to 1 when training.<p>
#@markdown Common tags that are removed such as hair color, etc. will be "absorbed" by your activation tag.
global_activation_tag = "" #@param {type:"string"}
remove_tags = "" #@param {type:"string"}
#@markdown &nbsp;

#@markdown In this advanced section, you can search text files containing matching tags, and replace them with less/more/different tags. If you select the checkbox below, any extra tags will be put at the start of the file, letting you assign different activation tags to different parts of your dataset. Still, you may want a more advanced tool for this.
search_tags = "" #@param {type:"string"}
replace_with = "" #@param {type:"string"}
search_mode = "OR" #@param ["OR", "AND"]
new_becomes_activation_tag = False #@param {type:"boolean"}
#@markdown These may be useful sometimes. Will remove existing activation tags, be careful.
sort_alphabetically = False #@param {type:"boolean"}
remove_duplicates = False #@param {type:"boolean"}

def split_tags(tagstr):
  return [s.strip() for s in tagstr.split(",") if s.strip()]

activation_tag_list = split_tags(global_activation_tag)
remove_tags_list = split_tags(remove_tags)
search_tags_list = split_tags(search_tags)
replace_with_list = split_tags(replace_with)
replace_new_list = [t for t in replace_with_list if t not in search_tags_list]

replace_with_list = [t for t in replace_with_list if t not in replace_new_list]
replace_new_list.reverse()
activation_tag_list.reverse()

remove_count = 0
replace_count = 0

for txt in [f for f in os.listdir(images_folder) if f.lower().endswith(".txt")]:

  with open(os.path.join(images_folder, txt), 'r') as f:
    tags = [s.strip() for s in f.read().split(",")]

  if remove_duplicates:
    tags = list(set(tags))
  if sort_alphabetically:
    tags.sort()

  for rem in remove_tags_list:
    if rem in tags:
      remove_count += 1
      tags.remove(rem)

  if "AND" in search_mode and all(r in tags for r in search_tags_list) \
      or "OR" in search_mode and any(r in tags for r in search_tags_list):
    replace_count += 1
    for rem in search_tags_list:
      if rem in tags:
        tags.remove(rem)
    for add in replace_with_list:
      if add not in tags:
        tags.append(add)
    for new in replace_new_list:
      if new_becomes_activation_tag:
        if new in tags:
          tags.remove(new)
        tags.insert(0, new)
      else:
        if new not in tags:
          tags.append(new)

  for act in activation_tag_list:
    if act in tags:
      tags.remove(act)
    tags.insert(0, act)

  with open(os.path.join(images_folder, txt), 'w') as f:
    f.write(", ".join(tags))

if global_activation_tag:
  print(f"\n📎 Applied new activation tag(s): {', '.join(activation_tag_list)}")
if remove_tags:
  print(f"\n🚮 Removed {remove_count} tags.")
if search_tags:
  print(f"\n💫 Replaced in {replace_count} files.")
print("\n✅ Done! Check your updated tags in the Extras below.")


## *️⃣ Extras

In [ ]:
if "step1_installed_flag" not in globals():
  raise Exception("Please run step 1 first!")

#@markdown ### 📈 Analyze Tags
#@markdown Perhaps you need another look at your dataset.
show_top_tags = 50 #@param {type:"number"}

from collections import Counter
top_tags = Counter()

for txt in [f for f in os.listdir(images_folder) if f.lower().endswith(".txt")]:
  with open(os.path.join(images_folder, txt), 'r') as f:
    top_tags.update([s.strip() for s in f.read().split(",")])

top_tags = Counter(top_tags)
print(f"📊 Top {show_top_tags} tags:")
for k, v in top_tags.most_common(show_top_tags):
  print(f"{k} ({v})")

In [ ]:
#@markdown ### 🔢 Count datasets
#@markdown Google Drive makes it impossible to count the files in a folder, so this will show you the file counts in all folders and subfolders.
folder = "/content/drive/MyDrive/Loras" #@param {type:"string"}

import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("📂 Connecting to Google Drive...\n")
    drive.mount('/content/drive')

tree = {}
exclude = ("_logs", "/output")
for i, (root, dirs, files) in enumerate(os.walk(folder, topdown=True)):
  dirs[:] = [d for d in dirs if all(ex not in d for ex in exclude)]
  images = len([f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))])
  captions = len([f for f in files if f.lower().endswith(".txt")])
  others = len(files) - images - captions
  path = root[folder.rfind("/")+1:]
  tree[path] = None if not images else f"{images:>4} images | {captions:>4} captions |"
  if tree[path] and others:
    tree[path] += f" {others:>4} other files"

pad = max(len(k) for k in tree)
print("\n".join(f"📁{k.ljust(pad)} | {v}" for k, v in tree.items() if v))


In [ ]:
#@markdown ##Repetitions Calculator ⌛📝
#@markdown Calculate the number of repetitions to use to train your Loras, `SD1.5` use batch of `1` and `2`, `SDXL and Pony` use a batch of `4`.
#@markdown If you use colab pro calculate your repetitions with `8` batch size (SDXL and Pony only).
# Variables
# Number of images
num_images = 40 # @param{type:"number"}
# Number of repetitions
num_repeats = 4 # @param{type:"number"}
# Number of epochs
num_epochs = 10 # @param{type:"number"}
# batch size
batch_size = 2 # @param{type:"number"}

# Calculate the result
resultado = (num_images * num_repeats * num_epochs) / batch_size

# Show the result
print("\33[96mEl total de repeticiones es:\033[0m", resultado)


In [ ]:
#@markdown ### 📂 Unzip dataset
#@markdown It's much slower to upload individual files to your Drive, so you may want to upload a zip if you have your dataset in your computer.
zip = "/content/drive/MyDrive/Loras/example.zip" #@param {type:"string"}
extract_to = "/content/drive/MyDrive/Loras/example/dataset" #@param {type:"string"}

import os, zipfile

if not os.path.exists('/content/drive'):
  from google.colab import drive
  print("📂 Connecting to Google Drive...")
  drive.mount('/content/drive')

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip, 'r') as f:
  f.extractall(extract_to)

print("✅ Done")


In [ ]:
if "step1_installed_flag" not in globals():
  raise Exception("Please run step 1 first!")

from PIL import Image
import os
Image.MAX_IMAGE_PIXELS = None

#@markdown ### 🖼️ Reduce dataset filesize
#@markdown This will convert all images in the project folder to jpeg, reducing filesize without affecting quality too much. This can also solve some errors.
location = images_folder

for dir in [d[0] for d in os.walk(location)]:
    os.chdir(dir)
    converted = False
    for file_name in list(os.listdir(".")):
        try:
            # Convert png to jpeg
            if file_name.endswith(".png"):
                if not converted:
                    print(f"Converting {dir}")
                    converted = True
                im = Image.open(file_name)
                im = im.convert("RGB")
                new_file_name = os.path.splitext(file_name)[0] + ".jpeg"
                im.save(new_file_name, quality=95)
                os.remove(file_name)
                file_name = new_file_name
            # Resize large jpegs
            if file_name.endswith((".jpeg", ".jpg")) and os.path.getsize(file_name) > 2000000:
                if not converted:
                    print(f"Converting {dir}")
                    converted = True
                im = Image.open(file_name)
                im = im.resize((int(im.width/2), int(im.height/2)))
                im.save(file_name, quality=95)
            # Rename jpg to jpeg
            if file_name.endswith(".jpg"):
                if not converted:
                    print(f"Converting {dir}")
                new_file_name = os.path.splitext(file_name)[0] + ".jpeg"
                os.rename(file_name, new_file_name)
        except Exception as e:
            print(f"An error occurred while processing {file_name}: {e}")
    if converted:
        print(f"Converted {dir}")
print(f"all images converted")


In [ ]:
if "step1_installed_flag" not in globals():
  raise Exception("Please run step 1 first!")

#@markdown ### 🚮 Clean folder
#@markdown Careful! Deletes all non-image files in the project folder.

!find {images_folder} -type f ! \( -name '*.png' -o -name '*.jpg' -o -name '*.jpeg' \) -delete
print(f"delete complete!")


In [ ]:
if "step1_installed_flag" not in globals():
  raise Exception("Please run step 1 first!")

import json
import time
from urllib.request import urlopen, Request

#@markdown ### 2️⃣ Scrape images from Gelbooru

#@markdown We will grab images from the popular anime gallery [Gelbooru](https://gelbooru.com/). Images are sorted by tags, including poses, scenes, character traits, character names, artists, etc. <p>
#@markdown * If you instead want to use your own images, upload them to your Google Drive's `Loras/project_name/dataset` folder.
#@markdown * If you instead want to download screencaps of anime episodes, try [this other colab by another person](https://colab.research.google.com/drive/1oBSntB40BKzNmKceXUlkXzujzdQw-Ci7). It's more complicated though.

#@markdown Up to 1000 images may be downloaded by this step in just one minute. Don't abuse it. <p>
#@markdown Your target tags should include the relevant tags for your character/concept/artstyle, and exclude undesired tags (for example, explicit images may affect learning).
#@markdown Separate words with underscores, separate tags with spaces, and use - to exclude a tag. You can also include a minimum score: `score:>10`
tags = "1girl -sex -greyscale -monochrome" #@param {type:"string"}
##@markdown If an image is bigger than this resolution a smaller version will be downloaded instead.
max_resolution = 3072 #param {type:"slider", min:1024, max:8196, step:1024}
##@markdown Posts with a parent post are often minor variations of the same image.
include_posts_with_parent = True #param {type:"boolean"}

tags = tags.replace(" ", "+")\
           .replace("(", "%28")\
           .replace(")", "%29")\
           .replace(":", "%3a")\
           .replace("&", "%26")\

url = "https://gelbooru.com/index.php?page=dapi&json=1&s=post&q=index&limit=100&tags={}".format(tags)
user_agent = "Mozilla/5.0 AppleWebKit/537.36 (KHTML, like Gecko; compatible; Googlebot/2.1; +http://www.google.com/bot.html) Chrome/93.0.4577.83 Safari/537.36"
limit = 100 # hardcoded by gelbooru
total_limit = 1000 # you can edit this if you want but I wouldn't recommend it
supported_types = (".png", ".jpg", ".jpeg")

def ubuntu_deps():
  print("🏭 Installing dependencies...\n")
  !apt -y install aria2
  return not get_ipython().__dict__['user_ns']['_exit_code']

if "step2_installed_flag" not in globals():
  if ubuntu_deps():
    clear_output()
    step2_installed_flag = True
  else:
    print("❌ Error installing dependencies, attempting to continue anyway...")

def get_json(url):
  with urlopen(Request(url, headers={"User-Agent": user_agent})) as page:
    return json.load(page)

def filter_images(data):
  return [p["file_url"] if p["width"]*p["height"] <= max_resolution**2 else p["sample_url"]
          for p in data["post"]
          if (p["parent_id"] == 0 or include_posts_with_parent)
          and p["file_url"].lower().endswith(supported_types)]

def download_images():
  data = get_json(url)
  count = data["@attributes"]["count"]

  if count == 0:
    print("📷 No results found")
    return

  print(f"🎯 Found {count} results")
  test_url = "https://gelbooru.com/index.php?page=post&s=list&tags={}".format(tags)
  display(Markdown(f"[Click here to open in browser!]({test_url})"))
  print (f"🔽 Will download to {images_folder.replace('/content/drive/', '')} (A confirmation box should appear below, otherwise run this cell again)")
  inp = input("❓ Enter the word 'yes' if you want to proceed with the download: ")

  if inp.lower().strip() != 'yes':
    print("❌ Download cancelled")
    return

  print("📩 Grabbing image list...")

  image_urls = set()
  image_urls = image_urls.union(filter_images(data))
  for i in range(total_limit // limit):
    count -= limit
    if count <= 0:
      break
    time.sleep(0.1)
    image_urls = image_urls.union(filter_images(get_json(url+f"&pid={i+1}")))

  scrape_file = os.path.join(config_folder, f"scrape_{project_subfolder}.txt")
  with open(scrape_file, "w") as f:
    f.write("\n".join(image_urls))

  print(f"🌐 Saved links to {scrape_file}\n\n🔁 Downloading images...\n")
  old_img_count = len([f for f in os.listdir(images_folder) if f.lower().endswith(supported_types)])

  os.chdir(images_folder)
  !aria2c --console-log-level=warn -c -x 16 -k 1M -s 16 -i {scrape_file}

  new_img_count = len([f for f in os.listdir(images_folder) if f.lower().endswith(supported_types)])
  print(f"\n✅ Downloaded {new_img_count - old_img_count} images.")

download_images()


In [ ]:
if "step1_installed_flag" not in globals():
  raise Exception("Please run step 1 first!")

#@markdown ### 3️⃣ Curate your images
#@markdown We will find duplicate images with the FiftyOne AI and delete them. <p>
#@markdown This is how similar 2 images must be to be marked for deletion. I recommend 0.97 to 0.99:
similarity_threshold = 0.985 #@param {type:"number"}
#@markdown You can choose between only deleting the duplicates, or additionally opening an interactive area below this cell that lets you visualize all your images and manually mark with `delete` to the ones you don't like. <p>
#@markdown If the interactive area appears blank for over a minute, try enabling cookies and removing tracking protection for the Google Colab website, as they may break it.
#@markdown Regardless, you can save your changes by sending Enter in the input box above the interactive area.<p>
action = "Delete duplicates" #@param ["Delete duplicates","Mark duplicates and open interactive area","Open interactive area"]
#@markdown To open the interactive area in a new tab INSTEAD of below, you need an ngrok account.
open_in_new_tab = False #@param {type:"boolean"}
ngrok_token = "" #@param {type:"string"}


os.chdir(root_dir)
model_name = "clip-vit-base32-torch"
supported_types = (".png", ".jpg", ".jpeg")
img_count = len(os.listdir(images_folder))
batch_size = min(250, img_count)

if "step3_installed_flag" not in globals():
  print("🏭 Installing dependencies...\n")
  !pip -q install fiftyone ftfy pyngrok
  !pip -q install fiftyone-db-ubuntu2204
  if not get_ipython().__dict__['user_ns']['_exit_code']:
    clear_output()
    step3_installed_flag = True
  else:
    print("❌ Error installing dependencies, attempting to continue anyway...")

os.environ["FIFTYONE_SERVER"] = "0"
import numpy as np
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F
from sklearn.metrics.pairwise import cosine_similarity
from pyngrok import ngrok, conf
from portpicker import pick_unused_port

non_images = [f for f in os.listdir(images_folder) if not f.lower().endswith(supported_types)]
if non_images:
  print(f"💥 Error: Found non-image file {non_images[0]} - This program doesn't allow it. Sorry! Use the Extras at the bottom to clean the folder.")
elif img_count == 0:
  print(f"💥 Error: No images found in {images_folder}")
else:
  print("\n💿 Analyzing dataset...\n")
  dataset = fo.Dataset.from_dir(images_folder, dataset_type=fo.types.ImageDirectory)
  if "duplicates" in action:
    model = foz.load_zoo_model(model_name)
    embeddings = dataset.compute_embeddings(model, batch_size=batch_size)

    batch_embeddings = np.array_split(embeddings, batch_size)
    similarity_matrices = []
    max_size_x = max(array.shape[0] for array in batch_embeddings)
    max_size_y = max(array.shape[1] for array in batch_embeddings)

    for i, batch_embedding in enumerate(batch_embeddings):
      similarity = cosine_similarity(batch_embedding)
      #Pad 0 for np.concatenate
      padded_array = np.zeros((max_size_x, max_size_y))
      padded_array[0:similarity.shape[0], 0:similarity.shape[1]] = similarity
      similarity_matrices.append(padded_array)

    similarity_matrix = np.concatenate(similarity_matrices, axis=0)
    similarity_matrix = similarity_matrix[0:embeddings.shape[0], 0:embeddings.shape[0]]

    similarity_matrix = cosine_similarity(embeddings)
    similarity_matrix -= np.identity(len(similarity_matrix))

    dataset.match(F("max_similarity") > similarity_threshold)
    dataset.tags = ["delete", "has_duplicates"]

    id_map = [s.id for s in dataset.select_fields(["id"])]
    samples_to_remove = set()
    samples_to_keep = set()

    for idx, sample in enumerate(dataset):
      if sample.id not in samples_to_remove:
        # Keep the first instance of two duplicates
        samples_to_keep.add(sample.id)

        dup_idxs = np.where(similarity_matrix[idx] > similarity_threshold)[0]
        for dup in dup_idxs:
            # We kept the first instance so remove all other duplicates
            samples_to_remove.add(id_map[dup])

        if len(dup_idxs) > 0:
            sample.tags.append("has_duplicates")
            sample.save()
      else:
        sample.tags.append("delete")
        sample.save()

    sidebar_groups = fo.DatasetAppConfig.default_sidebar_groups(dataset)
    for group in sidebar_groups[1:]:
      group.expanded = False
    dataset.app_config.sidebar_groups = sidebar_groups
    dataset.save()

  if "interactive" in action:
    clear_output()
    os.environ["FIFTYONE_SERVER"] = "1"
    port = pick_unused_port()
    session = fo.launch_app(dataset, port=port, auto=not open_in_new_tab)
    if open_in_new_tab:
      conf.get_default().auth_token = ngrok_token
      public_url = ngrok.connect(port).public_url
      print(f"🟢 Session open at {public_url}")

    print("❗ Wait a minute for the session to load. If it doesn't, read above.")
    print("❗ When it's ready, you'll see a grid of your images.")
    print("❗ On the left side enable \"sample tags\" to visualize the images marked for deletion.")
    print("❗ You can mark your own images with the \"delete\" label by selecting them and pressing the tag icon at the top.")
    input("⭕ When you're done, enter something here to save your changes: ")

    print("💾 Saving...")

  marked = [s for s in dataset if "delete" in s.tags]
  dataset.delete_samples(marked)
  previous_folder = images_folder[:images_folder.rfind("/")]
  dataset.export(export_dir=os.path.join(images_folder, project_subfolder), dataset_type=fo.types.ImageDirectory)

  temp_suffix = "_temp"
  !mv {images_folder} {images_folder}{temp_suffix}
  !mv {images_folder}{temp_suffix}/{project_subfolder} {images_folder}
  !rm -r {images_folder}{temp_suffix}

  if "interactive" in action:
    session.refresh()
    fo.close_app()
    clear_output()

  print(f"\n✅ Removed {len(marked)} images from dataset. You now have {len(os.listdir(images_folder))} images.")
